In [ ]:
import os
from ultralytics import YOLO

def train_helmet_model():
    # 1. Initialize a base YOLOv8 model. 
    # 'yolov8n.pt' (nano) or 'yolov8s.pt' (small) are ideal for real-time traffic edge deployment.
    model = YOLO('yolov8n.pt')

    # 2. Path to the configuration file we created in Step 1
    yaml_path = os.path.abspath('dataset.yaml')

    print(f"Starting YOLOv8 training using config: {yaml_path}")

    # 3. Kick off training
    model.train(
        data=yaml_path,     # Path to your dataset specification file
        epochs=50,          # Number of training epochs (adjust based on your timeline/GPU)
        imgsz=640,          # Standard image size for training and inference
        batch=16,           # Batch size (set to 8 or 32 depending on your GPU VRAM)
        device='mps',           # Use device=0 for CUDA GPU, or device='cpu' if you don't have a dedicated GPU
        workers=4,          # Number of CPU workers for loading data
        project='helmet_detection', # Saves runs to a folder named 'helmet_detection'
        name='yolov8_run'   # Subfolder name for this specific training session
    )


train_helmet_model()

In [ ]:
import cv2
from ultralytics import YOLO

# 1. Load your trained custom weights
model = YOLO('/Users/abishekkhadka/Desktop/Bachelor-s-Note/6th Sem/Automated-Two-Wheeler-Violation-Detection-and-E-Challan-System/runs/detect/helmet_detection/yolov8_run-4/weights/best.pt') 

# 2. Open the testing video
video_path = '/Users/abishekkhadka/Desktop/Bachelor-s-Note/6th Sem/Automated-Two-Wheeler-Violation-Detection-and-E-Challan-System/ML/Datasets/Helmet_kaggle/train/YTDown_YouTube_Indian-Traffic-Footage-Pixels_Media_4L_VloEggeo_001_1080p.mp4'
cap = cv2.VideoCapture(video_path)

# Get video properties for saving the output
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))

# Define code to save the output video
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('output_tracked_traffic2.mp4', fourcc, fps, (frame_width, frame_height))

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Run YOLO inference on the current frame
    # Classes: 0: with helmet, 1: without helmet, 2: rider, 3: number plate
    results = model(frame, conf=0.5) 

    # Visualize the results on the frame
    annotated_frame = results[0].plot()

    # Save and display the frame
    out.write(annotated_frame)
    cv2.imshow('Helmet & Number Plate Detection Test', annotated_frame)

    # Break loop if 'q' is pressed
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
out.release()
cv2.destroyAllWindows()
print("Testing complete. Output saved as 'output_tracked_traffic.mp4'")